In [1]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 105.8 MB/s eta 0:00:00


In [2]:
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git@main'

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.2/85.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 35.5 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%env CUDA_LAUNCH_BLOCKING=1

env: CUDA_LAUNCH_BLOCKING=1


In [5]:
import json
import os

dataset_path = "/content/drive/MyDrive/Vitra_chair.v3i.coco"

for split in ["train", "valid"]:
    with open(f"{dataset_path}/{split}/_annotations.coco.json") as f:
        data = json.load(f)

    # Rimappa category_id da 0 a 9
    for ann in data["annotations"]:
        ann["category_id"] = ann["category_id"] % 10

    # Rimuovi bbox non valide (larghezza/altezza <= 0)
    valid_annotations = []
    for ann in data["annotations"]:
        x, y, w, h = ann["bbox"]
        if w > 0 and h > 0:
            valid_annotations.append(ann)
        else:
            print(f"Rimosso bbox non valida in {split}:", ann)
    data["annotations"] = valid_annotations

    # Mantieni solo 10 categorie
    unique_cats = {cat["id"]: cat for cat in data["categories"]}.values()
    data["categories"] = list(unique_cats)[:10]

    # Salva JSON corretto
    with open(f"{dataset_path}/{split}/_annotations.coco.json", "w") as f:
        json.dump(data, f)

print("Dataset pulito e rimappato correttamente!")


Dataset pulito e rimappato correttamente!


In [6]:
for split in ["train","valid"]:
    with open(f"{dataset_path}/{split}/_annotations.coco.json") as f:
        data = json.load(f)

    cat_ids = [ann["category_id"] for ann in data["annotations"]]
    print(f"{split} category_id:", set(cat_ids))

    invalid_bbox = [ann for ann in data["annotations"] if ann["bbox"][2] <= 0 or ann["bbox"][3] <= 0]
    print(f"{split} bbox non valide trovate:", len(invalid_bbox))

train category_id: {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}
train bbox non valide trovate: 0
valid category_id: {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}
valid bbox non valide trovate: 0


In [7]:
from detectron2.data import DatasetCatalog, MetadataCatalog

# rimuove  registrazioni vecchie
for d in ["chairs_train", "chairs_val"]:
    if d in DatasetCatalog.list():
        DatasetCatalog.remove(d)
    if d in MetadataCatalog.list():
        MetadataCatalog.remove(d)

In [9]:
from detectron2.data.datasets import register_coco_instances

register_coco_instances(
    "chairs_train", {},
    f"{dataset_path}/train/_annotations.coco.json",
    f"{dataset_path}/train"
)

register_coco_instances(
    "chairs_val", {},
    f"{dataset_path}/valid/_annotations.coco.json",
    f"{dataset_path}/valid"
)

In [10]:
from detectron2.config import get_cfg
from detectron2 import model_zoo
import os

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml")

cfg.DATASETS.TRAIN = ("chairs_train",)
cfg.DATASETS.TEST = ("chairs_val",)
cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 10  # 10 modelli di sedie

cfg.SOLVER.IMS_PER_BATCH = 4
cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.MAX_ITER = 10000
cfg.OUTPUT_DIR = "/content/output"
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

In [11]:
from detectron2.engine import DefaultTrainer

trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

[11/27 09:38:41 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res

model_final_280758.pkl: 167MB [00:00, 224MB/s]                           
roi_heads.box_predictor.bbox_pred.{bias, weight}
roi_heads.box_predictor.cls_score.{bias, weight}


[11/27 09:38:42 d2.engine.train_loop]: Starting training from iteration 0


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W1127 09:38:46.651000 298 torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.


[11/27 09:39:01 d2.utils.events]:  eta: 2:03:25  iter: 19  total_loss: 2.567  loss_cls: 2.318  loss_box_reg: 0.214  loss_rpn_cls: 0.01295  loss_rpn_loc: 0.008925    time: 0.7441  last_time: 0.7576  data_time: 0.0502  last_data_time: 0.0131   lr: 4.9953e-06  max_mem: 3066M
[11/27 09:39:24 d2.utils.events]:  eta: 2:04:08  iter: 39  total_loss: 2.35  loss_cls: 2.103  loss_box_reg: 0.1544  loss_rpn_cls: 0.008155  loss_rpn_loc: 0.0087    time: 0.7481  last_time: 0.7585  data_time: 0.0245  last_data_time: 0.0125   lr: 9.9902e-06  max_mem: 3067M
[11/27 09:39:39 d2.utils.events]:  eta: 2:04:00  iter: 59  total_loss: 1.986  loss_cls: 1.751  loss_box_reg: 0.2108  loss_rpn_cls: 0.01415  loss_rpn_loc: 0.008911    time: 0.7453  last_time: 0.7738  data_time: 0.0135  last_data_time: 0.0115   lr: 1.4985e-05  max_mem: 3068M
[11/27 09:39:54 d2.utils.events]:  eta: 2:03:38  iter: 79  total_loss: 1.4  loss_cls: 1.164  loss_box_reg: 0.2135  loss_rpn_cls: 0.007135  loss_rpn_loc: 0.006968    time: 0.7452  la

In [12]:
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

# costruzione evaluator COCO sul validation set
evaluator = COCOEvaluator("chairs_val", cfg, False, output_dir="./output/")
val_loader = build_detection_test_loader(cfg, "chairs_val")

# Calcola tutte le metriche
metrics = inference_on_dataset(trainer.model, val_loader, evaluator)
print("Metriche di validation:")
print(metrics)


WARNING [11/27 11:47:39 d2.evaluation.coco_evaluation]: COCO Evaluator instantiated using config, this is deprecated behavior. Please pass in explicit arguments instead.
WARNING [11/27 11:47:39 d2.data.datasets.coco]: 
Category ids in annotations are not in [1, #categories]! We'll apply a mapping for you.

[11/27 11:47:39 d2.data.datasets.coco]: Loaded 615 images in COCO format from /content/drive/MyDrive/Vitra_chair.v3i.coco/valid/_annotations.coco.json
[11/27 11:47:39 d2.data.dataset_mapper]: [DatasetMapper] Augmentations used in inference: [ResizeShortestEdge(short_edge_length=(800, 800), max_size=1333, sample_style='choice')]
[11/27 11:47:39 d2.data.common]: Serializing the dataset using: <class 'detectron2.data.common._TorchSerializedList'>
[11/27 11:47:39 d2.data.common]: Serializing 615 elements to byte tensors and concatenating them all ...
[11/27 11:47:39 d2.data.common]: Serialized dataset takes 0.21 MiB
[11/27 11:47:39 d2.evaluation.evaluator]: Start inference on 615 batches

In [26]:
from detectron2.data import MetadataCatalog

metadata = MetadataCatalog.get("chairs_val")
bbox_metrics = metrics['bbox']

print("Class-wise simplified Precision/Recall (from AP50 / AR):\n")
for i, name in enumerate(metadata.thing_classes):
    ap50_key = f"AP-{name}"
    if ap50_key in bbox_metrics:
        precision = bbox_metrics[ap50_key] / 100  # AP50 è in %

        recall = bbox_metrics.get(f"AR-{name}", precision) / 100
        f1 = 2*precision*recall / (precision + recall + 1e-6)
        print(f"{name}: Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}")


Class-wise simplified Precision/Recall (from AP50 / AR):

objects: Precision=0.638, Recall=0.006, F1=0.013
APC: Precision=0.566, Recall=0.006, F1=0.011
Eames Plastic Chair: Precision=0.573, Recall=0.006, F1=0.011
Hal: Precision=0.641, Recall=0.006, F1=0.013
Mikado: Precision=0.721, Recall=0.007, F1=0.014
Mynt: Precision=0.635, Recall=0.006, F1=0.013
Panton: Precision=0.684, Recall=0.007, F1=0.014
Plywood: Precision=0.694, Recall=0.007, F1=0.014
Standard: Precision=0.674, Recall=0.007, F1=0.013
TipTon: Precision=0.589, Recall=0.006, F1=0.012


In [30]:
print(model)

GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res2): Sequential(
        (0): BottleneckBlock